# Chapter08
+ 아래 install 및 openai key입력을 진행해주세요
+ 파일이 colab 혹은 폴더 path에 넣어져 있는지 확인해주세요

In [12]:
! pip install langchain==1.2.14 \
    langchain_openai==1.1.12 \
    langchain_community==0.4.1 \
    pymupdf \
    langchain-text-splitters \
    docling \
    langchain_chroma==1.1.0 \
    gradio==5.49.1 \
    rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: tomlkit
    Found existing installation: tomlkit 0.14.0
    Uninstalling tomlkit-0.14.0:
      Successfully uninstalled tomlkit-0.14.0
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.4
    Uninstalling pydantic_core-2.46.4:
      Successfully uninstalled pydantic_core-2.46.4
  Attempting uninstall: aiofiles
    Found existing installation: aiofiles 25.1.0
    Uninstalling aiof

설치 후 런타임 재시작(Colab 상단 메뉴 → 런타임 → 세션 다시 시작)

In [2]:
import getpass
import os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


## 8.6 [프로젝트] 금융 PDF를 이용한 RAG 챗봇

In [3]:
!wget -O "한국은행 2024년 연차보고서.pdf" "https://raw.githubusercontent.com/freelec-llm-scientist/freelec_llm_scientist_code/main/chapter_08/content/한국은행 2024년 연차보고서.pdf"
!wget -O "한국은행 2025년 9월중 금융시장 동향.pdf" "https://raw.githubusercontent.com/freelec-llm-scientist/freelec_llm_scientist_code/main/chapter_08/content/한국은행 2025년 9월중 금융시장 동향.pdf"

--2026-07-06 13:46:25--  https://raw.githubusercontent.com/freelec-llm-scientist/freelec_llm_scientist_code/main/chapter_08/content/%ED%95%9C%EA%B5%AD%EC%9D%80%ED%96%89%202024%EB%85%84%20%EC%97%B0%EC%B0%A8%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11549090 (11M) [application/octet-stream]
Saving to: ‘한국은행 2024년 연차보고서.pdf’

한국은행 2024년 연  100%[===================>]  11.01M  --.-KB/s    in 0.1s    

2026-07-06 13:46:25 (114 MB/s) - ‘한국은행 2024년 연차보고서.pdf’ saved [11549090/11549090]

--2026-07-06 13:46:25--  https://raw.githubusercontent.com/freelec-llm-scientist/freelec_llm_scientist_code/main/chapter_08/content/%ED%95%9C%EA%B5%AD%EC%9D%80%ED%96%89%202025%EB%85%84%209%EC%9B%94%EC%A4%91%20%EA%B8%88%EC%9C%B5%EC%8B%9C%EC%9E%A5%20

In [4]:
# 한국은행 2024년 연차보고서.pdf
from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker.hierarchical_chunker import HierarchicalChunker
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
import os
import getpass

pdf_path = "/content/한국은행 2024년 연차보고서.pdf"
converter = DocumentConverter()
result = converter.convert(source=pdf_path)
doc = result.document

[INFO] 2026-07-06 13:46:44,083 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-06 13:46:44,115 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-06 13:46:44,117 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-06 13:46:44,189 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-06 13:46:44,194 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-06 13:46:44,195 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-06 13:46:44,253 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-06 13:46:44,317 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[WARNING] 2026-07-06 13:46:51,817 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-07-06 13:48:18,092 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-07-06 13:49:09,765 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-07-06 13:49:11,460 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-07-06 13:50:16,931 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-07-06 13:51:22,056 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-07-06 13:51:52,403 [RapidOCR] main.py:132: The text detection result is empty


In [5]:
# 구조 기반 청킹
hc = HierarchicalChunker()
chunks = list(hc.chunk(dl_doc=doc))
print(f"청크 수: {len(chunks)}")

print(f"총 {len(chunks)}개의 청크 생성 완료.")
if chunks:
    print(chunks[0].text[:300])

# API 키 설정
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

# 임베딩 모델 초기화
embedding = OpenAIEmbeddings(model="text-embedding-3-large")

# 벡터DB 구축
vectorstore = Chroma.from_texts(
    [c.text for c in chunks],
    embedding=embedding,
    collection_name="finance_docs"
)

print(f"Chroma 컬렉션에 {vectorstore._collection.count()}개의 문서 임베딩 완료")

청크 수: 1309
총 1309개의 청크 생성 완료.
2025. 3
Chroma 컬렉션에 1309개의 문서 임베딩 완료


In [6]:
! pip install rank_bm25

In [7]:
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever

# BM25 검색기 (텍스트 유사도)
bm25_retriever = BM25Retriever.from_texts([c.text for c in chunks])
# 벡터 검색기
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
# 하이브리드 검색 결합
retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]
)


In [8]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. LLM 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

# 2. 프롬프트 정의 (최신 방식에서는 프롬프트 작성이 필수이자 핵심입니다)
system_prompt = (
    "주어진 문맥(context)을 사용하여 사용자의 질문에 답하세요. "
    "만약 답을 모른다면, 모른다고 정직하게 말하세요."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 3. 최신 조립형 체인 구축 (RetrievalQA 완벽 대체)
# Step A: 검색된 문서(context)를 LLM에 어떻게 밀어넣을지 결정 (stuff 방식)
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# Step B: 리트리버(검색기)와 앞서 만든 체인을 결합
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

# 4. 실행 및 결과 출력 (.run() 대신 .invoke() 사용)
query = "양자 간 통화 스왑 가능 국가를 전부 알려주세요"

# 최신 체인은 딕셔너리 형태로 입출력을 받습니다.
response = rag_chain.invoke({"input": query})

# 결과 출력 (정답 텍스트는 'answer' 키 안에 담겨서 옵니다)
print("답변:", response['answer'])

답변: 양자 간 통화 스왑 가능 국가는 다음과 같습니다:

1. 캐나다
2. 중국
3. 스위스
4. 일본
5. 인도네시아
6. 호주
7. UAE (아랍에미리트)
8. 말레이시아
9. 튀르키예

이들 국가와 한국은행 간에 통화 스왑 계약이 체결되어 있습니다.


In [16]:
import gradio as gr

def chat_fn(message, history):
    history = history + [{"role": "user", "content": message}]
    answer = rag_chain.invoke({"input": message})["answer"]
    history = history + [{"role": "assistant", "content": answer}]
    return history, ""

with gr.Blocks(title="💰 금융 보고서 RAG 챗봇") as demo:
    gr.Markdown("## 💰 금융 보고서 RAG 챗봇")
    chatbot = gr.Chatbot(type="messages", height=450)
    msg = gr.Textbox(placeholder="질문을 입력하세요", show_label=False)

    # 채팅창 아래에 예시 버튼 고정
    gr.Examples(
        examples=["양자 간 통화 스왑 가능 국가를 전부 알려주세요"],
        inputs=msg,
    )

    msg.submit(chat_fn, [msg, chatbot], [chatbot, msg])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://570ad8723101984976.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
